In [1]:
import torch
import json
from PIL import Image
from transformers import DonutProcessor, VisionEncoderDecoderModel,Seq2SeqTrainingArguments, Seq2SeqTrainer,default_data_collator
import pandas as pd
import numpy as np
import os
from torch.utils.data import Dataset
import tqdm
from google.colab import drive

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


Run this to check if Donut Works on your machine

In [3]:
print(f"Torch version: {torch.__version__}")

try:
    from transformers import DonutProcessor
    print("DonutProcessor imported successfully!")
except ImportError as e:
    print(f"Import failed: {e}")

Torch version: 2.10.0+cu128
DonutProcessor imported successfully!


Loading data

In [4]:
common_path = os.path.join("/content/drive/MyDrive","Train licenta","Train Dataset")
dataset_path = os.path.join(common_path,"batch_1_images")
label_path = os.path.join(common_path,"batch1_1.csv")
df_labels = pd.read_csv(label_path)
df_labels.sort_values(by='File Name',ascending=True,inplace=True)
df_labels.set_index(df_labels['File Name'].map(lambda name: int(name[7:11])).values,inplace=True)

Defining and loading a pretrained model

In [ ]:
processor = DonutProcessor.from_pretrained("naver-clova-ix/donut-base-finetuned-cord-v2")
model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base-finetuned-cord-v2")
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

Proper Donut Parser Class

In [6]:
class DonutInvoiceParser(Dataset):
    def __init__(self,dataset_path,df_labels,processor,max_length=512):
        self.dataset_path = dataset_path
        self.processor = processor
        self.max_length = max_length
        self.df_labels = df_labels

    def __len__(self):
        return len(self.df_labels)

    def __getitem__(self, idx):
        row = self.df_labels.iloc[idx]
        file_name = f"batch1-{idx:04d}.jpg"
        # 1. Load Image
        image_path = os.path.join(self.dataset_path,file_name)
        image = Image.open(image_path).convert("RGB")
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()

        # 2. Process Labels (Feedback)
        # We turn the JSON metadata into a string the decoder can learn
        json_data = json.loads(row['Json Data'])
        gt_dict = {"gt_parse":json_data}
        gt_string = json.dumps(gt_dict)
        labels = self.processor.tokenizer(
            gt_string,
            add_special_tokens=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze()

        # We use -100 to ignore padding in the loss calculation
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {"pixel_values": pixel_values, "labels": labels}


In [7]:
train_dataset = DonutInvoiceParser(dataset_path=dataset_path,df_labels=df_labels,processor=processor,max_length=512)

In [13]:
new_tokens = ["<invoice_no>", "<total>", "<invoice_date>", "<due_date>", "<Qty>"]
processor.tokenizer.add_tokens(new_tokens)
model.decoder.resize_token_embeddings(len(processor.tokenizer))

# Set essential generation parameters for the model config
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.bos_token_id

In [14]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./donut_invoice_results",
    per_device_train_batch_size=1,     # Keep at 1 for local machines to avoid memory errors
    gradient_accumulation_steps=8,     # This simulates a batch size of 8
    learning_rate=5e-5,                # Standard starting point for fine-tuning
    num_train_epochs=3,                # How many times to see the whole dataset
    logging_steps=10,
    save_steps=100,                    # Save a checkpoint every 100 steps
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),    # Use mixed precision if you have a GPU
    optim="adamw_torch",
    remove_unused_columns=False,
)

In [15]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,       # This is the class we built in the previous step
    data_collator=default_data_collator,
)

In [ ]:
trainer.train()

Step,Training Loss
